<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP7.1B/MP7.1B_insulation_ageing.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP7.1B · How Old Is the Insulation?

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'problem.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/")
PROJECT = (['mp71b.py', 'MP7.1B_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP7.1B/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex07.1-poisson", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex07.1-poisson", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'problem', 'mp71b']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import problem as hs
import mp71b as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP7.1B_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

pb = hs                                                  # Ex07.1's problem.py, by its name in Ex07.1

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

The insulation of a winding ages. Held for years near its rated temperature, the impregnating resin between the wires cracks and comes away from the copper, and air - 0.03 W/(m·K) against the resin's 0.2 - takes its place. Where that happens the winding conducts heat worse, the copper next to it runs hotter, and the insulation ages faster still. An operator cannot open the machine to look; what can be done is to read the temperature at a few points in the slot (thermocouples or fibre-optic probes embedded when the machine is wound) at a few currents, and **infer from the readings where the insulation has aged, how much, and how hot the slot's hot spot now runs**.

This is an **inverse problem**: Ex07.1 computed the temperature from a known slot; here the slot is partly unknown and the temperatures are measured. And it is asked **again and again** - for every machine of a fleet, at every inspection - so what matters is not only the answer but the time to it.

## A2 · The slot, and what is new against Ex07.1

The slot is Ex07.1's: 10 × 20 mm, its 32 wires averaged into one material, $k_\mathrm{eff} = 0.70$ W/(m·K), with the heat $q = 0.503\,\rho(T)\,J^2$ spread over it, the walls held at 90 °C. **New:** somewhere in it a disc of **aged insulation** - centre $(x_d, y_d)$, radius $r_d$ - has a lower conductivity $k_d$. Six sensors read the rise at four currents, with noise.

| | value |
|---|---|
| the slot | 10 × 20 mm, $k_\mathrm{eff}$ = 0.70 W/(m·K), walls at 90 °C (Ex07.1) |
| the aged disc | centre anywhere with the disc inside the slot; radius 1.0 to 2.5 mm; $k_d$ = 0.28 to 0.49 W/(m·K), 40 to 70 % of $k_\mathrm{eff}$ (assumed range; "halved" is 0.35) |
| the sensors | six, at $x = \pm 2$ mm and $y = -4.5, 0, 4.5$ mm; in the harder cases the top pair is missing (four sensors) |
| the currents | 15, 25, 31.42 (rated) and 40 A per wire |
| the noise | 0.05 or 0.10 K, one standard deviation, independent at every reading (assumed for thermocouples) |

**Why no exact solution exists.** Ex07.1's rectangle has constant coefficients and an exact series. A disc where $k$ jumps breaks it: no mode of the rectangle solves the equation across the disc's edge. **The strength of the change** is $1 - k_d/k_\mathrm{eff}$: at $k_d = k_\mathrm{eff}$ there is no disc, and the solver must reproduce Ex07.1's exact series (part C checks it).

In [ ]:
# The slot with a disc of aged insulation, the six sensors, and the readings of test case 1
G = np.load("MP7.1B_truth.npz")
fig, ax = plt.subplots(1, 2, figsize=(10.5, 5.6), gridspec_kw={"width_ratios": [1, 1.6]})
mp.draw_slot(ax[0], disc=G["case1_disc"])                   # the slot, the hidden disc of case 1, the sensors
I = G["current"]
for j in range(6):
    ax[1].plot(I, G["case1_readings"][:, j] - G["healthy_readings"][:, j], "o-", label=f"sensor {j + 1}")
ax[1].set_xlabel("current I [A]"); ax[1].set_ylabel("reading - healthy slot's [K]"); ax[1].legend(fontsize=8)
ax[1].set_title("what the disc changes at each sensor (case 1, with its noise)", fontsize=10)
plt.tight_layout(); plt.show()
print("sensors [mm]:", np.round(G["sensors"] * 1e3, 1).tolist())
print("readings of case 1 [K], one row per current:"); print(np.round(G["case1_readings"], 2))
print("the healthy slot at the same sensors [K]:");   print(np.round(G["healthy_readings"], 2))

**What you should see.** Left, the slot with case 1's hidden disc drawn (you are shown it here; the readings are what you fit) and the six sensors. Right, how much each sensor reads above the healthy slot at each current, noise included. Case 1's disc raises sensor 4 by about 0.08 K at 15 A and 0.68 K at 40 A (without the noise) - growing with the current as about $I^2$ - and lowers sensor 1, on the far side, by 0.01 to 0.09 K, about its noise: the heat the disc holds back goes round it. The noise of 0.05 K is comparable with the smaller changes: **the disc is seen, but not clearly**, and that sets what can be inferred.

## A3 · The equation and its conditions

$$\nabla\cdot\big(k(x, y)\,\nabla\theta\big) + q(\theta, I) = 0 \quad\text{in the slot}, \qquad k = \begin{cases} k_d & \text{inside the disc}\\ k_\mathrm{eff} & \text{outside}\end{cases}, \qquad q = 0.503\,(c_0 + c_1\theta)$$

| where | condition | why it holds |
|---|---|---|
| the four walls | $\theta = 0$ | the cooled iron; Ex07.1's condition |
| the disc's edge | $\theta$ and $k\,\partial\theta/\partial n$ continuous | no gap and no heat stored at the edge |
| the sensors | $\theta(x_s, y_s; I) = $ reading $\pm$ noise | the measurements, at six (or four) points and four currents |

The disc does not make heat: the aged insulation is in the bundle, between the wires, and the averaged slot spreads the copper's heat everywhere.

## A4 · Why the inverse problem is hard

* **Few readings.** 24 readings (16 with four sensors) for four unknowns, each reading a weighted average of the whole field.
* **A weak signal.** The disc changes the readings by tenths of a kelvin against a noise of 0.05 to 0.1 K.
* **Radius and conductivity trade off.** Seen from a distance, a disc of lower conductivity changes the field as if a single quantity mattered: its area times its contrast, $\pi r_d^2\,(k_\mathrm{eff} - k_d)/(k_\mathrm{eff} + k_d)$, called here the disc's **strength** [mm²]. A large, mildly aged disc and a small, strongly aged one look almost alike to sensors far from them. The strength is therefore asked within a tolerance, and the radius and conductivity apart are reported but not required.
* **The quarter is ambiguous.** The slot is symmetric, so a disc and its mirror image give readings that differ only through the sensors' asymmetric view of them.

## A5 · What is asked

Five test cases of rising difficulty, all public in `MP7.1B_truth.npz` - the readings to fit, and the hidden answers to score against afterwards:

| case | sensors | noise [K] |
|---|---|---|
| 1 | six | 0.05 |
| 2 | six | 0.10 |
| 3 | six | 0.10 |
| 4 | four (the top pair missing) | 0.05 |
| 5 | four | 0.10 |

| quantity | required, in every case |
|---|---|
| the disc's centre | within **1.2 mm** |
| the disc's strength $\pi r_d^2 (k_\mathrm{eff}-k_d)/(k_\mathrm{eff}+k_d)$ | within **35 %** |
| the hot spot of the slot at rated current | within **0.3 K** |
| the time to answer a new set of readings | at least **10 times shorter** than a new least-squares fit with the reference model (part C) |

One network, trained once over the whole range of discs, answers a new set of readings by being evaluated: **amortised** inference. Its training is reported apart, as the offline cost.

**The agreed accuracy, and why it differs from the first proposal.** The proposal asked for the disc within 0.5 mm, its $k$ within 20 % and the hot spot within 0.1 K. Part C shows that with these sensors and this noise **no method can promise that**: the information the readings carry (the Cramér-Rao bound of part C) leaves a standard deviation of 0.21 to 0.71 mm on the centre, 12 to 36 % on the strength and 0.23 to 0.35 K on the hot spot, whatever the method; the least-squares fit with the reference model misses case 1's centre by 1.0 mm and the hot spot by up to 0.29 K. The tolerances above are about one and a half to two of those standard deviations at the hardest case, and both the least-squares reference and the solved network meet them on all five cases. The radius and the conductivity apart are reported, not required (section 4).

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

An amortised network inverts the readings directly:

$$\text{readings } (4 \times 6) \text{ and which sensors are present } \longmapsto\ (x_d,\ y_d,\ r_d,\ k_d,\ \text{strength},\ \text{hot spot at each current})$$

Trained on many slots whose disc is known, it learns the map from readings to disc once. A new set of readings then costs one evaluation, against the many forward solves a least-squares fit needs.

Its physics comes from the **forward model**: every training example is a slot solved by the finite-volume model of part C at a random disc, its readings at the six sensors and its hot spot. Noise is added afresh at every step of training, so the network learns to read through it. A **physics-informed** alternative - an inverse PINN for $\theta(x, y, I)$ and $k(x, y)$ fitted to one set of readings and the equation - answers one set of readings at a time and must be trained again for every new set; you may compare it in question 9.

The scales of the problem:

In [ ]:
# The scales: the readings, and how much a disc changes them
s = mp.Slot(mp.REF_H)                                       # the reference model's grid (part C)
healthy, hot0 = s.readings(None)                            # the slot with no disc
print("healthy readings at rated current [K]:", np.round(healthy[2], 2), f"| hot spot {hot0[2]:.2f} K")
for d in ([0.0, -4.5e-3, 2.5e-3, 0.28], [0.0, -4.5e-3, 1.0e-3, 0.49], [1.0e-3, 2.0e-3, 1.75e-3, 0.35]):
    r, h = s.readings(np.array(d))
    print(f"disc at ({d[0]*1e3:4.1f}, {d[1]*1e3:4.1f}) mm, r {d[2]*1e3:.2f} mm, k {d[3]:.2f}: strength {mp.strength(d[2], d[3]):5.2f} mm², "
          f"largest change of a reading {np.abs(r - healthy).max():.2f} K, hot spot +{h[2] - hot0[2]:.2f} K")

**What you should see.** The healthy slot reads 13.98 to 15.77 K at rated current at the six sensors; its hot spot is 18.74 K, Ex07.1's. The strongest disc of the range, 2.5 mm and $k$ = 0.28 under a sensor (strength 8.41 mm²), changes a reading by 1.94 K and raises the hot spot by 2.72 K; the weakest, 1 mm and 0.49 (0.55 mm²), changes a reading by 0.05 K - the noise - and the hot spot by 0.04 K. The **strength** orders the discs by how much they are seen.

## B2 · The physics the training data obey

The forward model solves part A's equation for each disc. Two balances hold in every solution and are checked in part C: the heat made in the slot leaves through its walls, and at $k_d = k_\mathrm{eff}$ the field is Ex07.1's exact series. The readings are bilinear interpolations of the field at the sensors, the hot spot its largest cell value.

| what | how it enters |
|---|---|
| the heat equation with the disc | each training example is one forward solve, so every example satisfies it |
| the noise | added at every training step, $\sigma$ drawn from 0.05 to 0.10 K |
| missing sensors | in half the examples the top pair is set to zero and flagged, so one network serves six and four sensors |
| the hot spot | a target beside the disc's parameters: it is what the operator needs, and it is a smoother function of the readings than the disc's radius is |

## B3 · The training data

`mp.sample_discs(n, seed)` draws discs uniformly over the range; `mp.simulate(discs)` solves each one on the reference grid and returns its clean readings (n, 4 currents, 6 sensors) and its hot spot at each current. 15 to 40 ms per disc on one CPU core, depending on what else the machine runs: 20 000 discs took 5 to 12 minutes in our runs, a cost paid once, offline, like the training. The cell below simulates a few to show them; part D simulates the training set.

In [ ]:
# A few training examples: discs drawn over the range, and their readings
D = mp.sample_discs(400, seed=1)
t0 = time.perf_counter()
Rd, Hd = mp.simulate(D, verbose=False)
print(f"{len(D)} discs in {time.perf_counter() - t0:.1f} s")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
ax[0].scatter(D[:, 0] * 1e3, D[:, 1] * 1e3, c=mp.strength(D[:, 2], D[:, 3]), s=8, cmap="viridis")
ax[0].scatter(mp.SENSORS[:, 0] * 1e3, mp.SENSORS[:, 1] * 1e3, marker="s", c="r", label="sensors")
ax[0].set_aspect("equal"); ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("y [mm]"); ax[0].legend(fontsize=8)
ax[0].set_title("disc centres, coloured by strength", fontsize=9)
ax[1].scatter(mp.strength(D[:, 2], D[:, 3]), np.abs(Rd - healthy).max((1, 2)), s=4)
ax[1].axhline(0.1, color="r", lw=0.6); ax[1].set_xlabel("strength [mm²]"); ax[1].set_ylabel("largest change of a reading [K]")
ax[1].set_yscale("log")
ax[2].scatter(mp.strength(D[:, 2], D[:, 3]), Hd[:, 2] - hot0[2], s=4)
ax[2].set_xlabel("strength [mm²]"); ax[2].set_ylabel("hot spot rise, rated current [K]")
plt.tight_layout(); plt.show()
print(f"share of discs that change some reading by more than 0.1 K: {(np.abs(Rd - healthy).max((1, 2)) > 0.1).mean()*100:.0f} %")

**What you should see.** 400 discs simulated in 6 to 17 s. Left, their centres over the slot and the six sensors. Middle, the largest change a disc makes to any reading, against its strength: it grows with the strength, but at the same strength it varies several times with where the disc is. The red line is the larger noise, 0.1 K; 99 % of the discs change some reading by more than that, at 40 A. Right, how much the disc raises the hot spot at rated current: up to about 3 K, most for a strong disc near the slot's centre, where the slot is hottest.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Three solvers, three jobs

| | the **forward model** | the **reference model** | the **ground truth** |
|---|---|---|---|
| what it is | finite volumes on the whole slot | least squares over the disc's four parameters, the forward model on the engineering grid inside | the forward model on a fine grid |
| grid | any | 40 × 80 cells of 0.25 mm | 400 × 800 cells of 0.025 mm |
| its job | to make the training data | to be **timed** against the network: the online cost of a new set of readings | to make the test cases' readings and to **score** the answers |

## C2 · The forward model, step by step

1. **Cells.** The slot is cut into square cells of side $h$; the unknown is the rise in each.
2. **The disc in the cells.** A cell the disc's edge crosses gets the share $f$ inside the disc from a ramp one cell wide across the edge, and the conductivity $1/(f/k_d + (1-f)/k_\mathrm{eff})$. The readings then change smoothly as the disc moves or grows - a disc that snapped from cell to cell would give least squares a staircase to descend.
3. **Heat between neighbours** through the harmonic mean of their conductivities, so the heat flow is the same on both sides of every face; **to the walls** through half a cell.
4. **The heat's growth with the rise.** $q = c_0 + c_1\theta$, with $c_1/c_0 = 0.3$ %: the conductance matrix is factorised once per disc, and four fixed-point steps on that one factorisation take in the growth (each reduces the remaining error about 25 times).
5. **Readings.** The field at each sensor by bilinear interpolation between cell centres, the walls at zero included.

## C3 · The reference model: least squares

`mp.least_squares_fit(readings, sensors, noise)` finds the disc whose predicted readings are closest to the measured ones, each residual divided by the noise, with `scipy.optimize.least_squares` (trust-region reflective, within the range's bounds, derivatives by finite differences). The readings do not tell in which quarter of the slot the disc lies, so it starts from **four** discs, one in each quarter, and keeps the best fit: a few hundred forward solves per set of readings.

In [ ]:
# The ground truth: convergence on case 1's disc at rated current, from the truth file
G = np.load("MP7.1B_truth.npz")
print("cell [mm]      cells     hot spot [K]   sensor 4 [K]   energy balance off by")
for h, n, t, s4, b in zip(G["conv_h_mm"], G["conv_cells"], G["conv_hot"], G["conv_sensor4"], G["conv_balance"]):
    print(f"{h:8.4f}  {int(n):9d}    {t:10.4f}     {s4:10.4f}       {abs(b):.1e}")
d = np.abs(np.diff(G["conv_hot"]))
print("change of the hot spot between grids [K]:", np.round(d, 4))
print(f"at k_d = k_eff (no disc) against Ex07.1's exact series: {float(G['zero_strength_error_K']):.1e} K at most")

**What you should see.** Four grids, each twice as fine, on case 1's disc. The hot spot changes by 0.0040, 0.0008 and 0.0007 K, sensor 4 by 0.0011, 0.0006 and 0.0004 K. The ground truth (0.025 mm cells) is within 0.0007 K of the finest grid: four hundred times finer than the 0.3 K asked of the hot spot and seventy times finer than the 0.05 K noise. The order is about one, not two: the disc's edge, ramped over one cell, adds an error proportional to the cell size, so Richardson extrapolation at order two is not used. The heat made equals the heat through the walls to $7\times10^{-8}$ (the residual of the four fixed-point steps), and with no disc the same code gives Ex07.1's exact series to $2\times10^{-4}$ K.

In [ ]:
# The five test cases: what the readings hold, and the least-squares reference on each
CASES = [str(c) for c in G["cases"]]
slot = mp.Slot(mp.REF_H)                                     # the engineering grid, built once
LSQ = {}
print("case  sensors noise  | centre error [mm]  strength error  hot spot error at rated [K] | forward solves  time [s]")
for n in CASES:
    m = G[f"case{n}_mask"].astype(bool)                       # which sensors were read
    noise = float(G[f"case{n}_noise"])
    f = mp.least_squares_fit(G[f"case{n}_readings"][:, m], G["sensors"][m], noise, slot=slot)
    d = G[f"case{n}_disc"]
    e_c = np.hypot(*(f["disc"][:2] - d[:2])) * 1e3
    e_s = mp.strength(f["disc"][2], f["disc"][3]) / mp.strength(d[2], d[3]) - 1
    e_h = f["hot"][2] - G[f"case{n}_hot"][2]
    LSQ[n] = dict(f, centre_mm=e_c, strength=e_s, hot=e_h)
    print(f"  {n}     {m.sum()}     {noise:.2f}  |      {e_c:5.2f}            {e_s*100:+5.0f} %          {e_h:+.2f}"
          f"              |      {f['solves']:4d}        {f['seconds']:5.1f}")
t_lsq = float(np.median([v["seconds"] for v in LSQ.values()]))
REF = {"t_ref_s": t_lsq, "lsq": LSQ}                         # kept for part E
print(f"\nleast squares: {t_lsq:.1f} s per new set of readings (the median of the five); "
      f"the network must answer in under {t_lsq*1e3/10:.0f} ms")

**What you should see.** The least-squares reference finds the centre within 0.22 to 1.00 mm, the strength within 15 % and the hot spot at rated current within 0.29 K, in 243 to 897 forward solves, 4 to 39 s per set of readings on one CPU core (the median 5.9 s alone on the machine, 16.9 s when it shared the cores; this runtime's median is kept in `REF`). Case 1, at the smaller noise, has the largest centre error: with the noise drawn, a slightly larger and stronger disc 1 mm lower fits the readings as well as the true one, and it puts the hot spot 0.29 K low - the trade-off of part A, section 4. The requirements of part A were set from these numbers and the bound below: the reference meets them on all five cases.

In [ ]:
# What the readings can tell: the Cramér-Rao bound at each case's true disc
print("case | smallest possible standard deviation of: centre [mm]  strength [%]  hot spot at rated [K]")
for n in CASES:
    m = G[f"case{n}_mask"].astype(bool)
    c, s_, h = mp.information_bound(G[f"case{n}_disc"], G["sensors"][m], float(G[f"case{n}_noise"]), slot=slot)
    print(f"  {n}  |                                    {c:5.2f}         {s_:4.0f}            {h:5.2f}")

**What you should see.** The **Cramér-Rao bound**: from how fast the readings change as each parameter of the disc changes (the Jacobian $J$, by finite differences) and the noise $\sigma$, the covariance $\sigma^2 (J^\top J)^{-1}$ is the smallest any unbiased method can reach. It is a statement about the readings, not about a method. One standard deviation: the centre 0.21 to 0.71 mm, the strength 12 to 36 %, the hot spot 0.23 to 0.35 K. A single answer can be one or two of these off. A network trained over the range of discs can do somewhat better than the bound where it has learnt which discs are likely - it is not unbiased - but not much.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2, including how many slots you simulate), and the **figures** (TODO 3).

**The contract.** Your network is a PyTorch module called `net`, used as `net(readings, mask)`:

| argument | meaning | shape |
|---|---|---|
| `readings` | the 24 readings, rise above the wall [K], one row per current (15, 25, 31.42, 40 A), one column per sensor; zero where a sensor is missing | `(n, 4, 6)` |
| `mask` | 1 for a sensor that was read, 0 for a missing one | `(n, 6)` |
| returns | $x_d$, $y_d$, $r_d$ [m], $k_d$ [W/(m·K)], strength [mm²], hot spot at the four currents [K] | `(n, 9)` |

Whatever you compute inside - features of the readings, scaling, several heads - stays inside `net`.

## D2 · The training data and the loss, given

`make_data(n, seed)` simulates n slots with the forward model (part B) and returns the clean readings and the targets in the contract's order; `noisy(readings, sigma_range, four_share, generator)` adds fresh noise and removes the top pair of sensors in a share of the examples. `loss(net, readings, mask, targets)` is the mean squared error of the nine outputs, each divided by its range so that all count alike.

In [ ]:
# The training data and the loss, given ---------------------------------------------
HEALTHY = torch.tensor(mp.Slot(mp.REF_H).readings(None)[0], device=DEVICE)   # (4, 6) readings of the healthy slot [K]
SCALE = torch.tensor([5e-3, 10e-3, 0.75e-3, 0.105, 5.0, 3.0, 3.0, 3.0, 3.0], device=DEVICE)   # the outputs' ranges
CENTRE = torch.tensor([0, 0, 1.75e-3, 0.385, 2.0, 5.0, 12.0, 20.0, 33.0], device=DEVICE)       # and their middles

def make_data(n, seed):
    """n slots with random discs, simulated: readings (n, 4, 6) [K] and targets (n, 9), as tensors."""
    D = mp.sample_discs(n, seed)
    R_, H_ = mp.simulate(D, verbose=True)
    targets = np.c_[D, mp.strength(D[:, 2], D[:, 3]), H_]
    return torch.tensor(R_, device=DEVICE), torch.tensor(targets, device=DEVICE)

def noisy(readings, sigma=(0.05, 0.10), four_share=0.5, generator=None):
    """The readings with fresh noise (a σ drawn per example) and, in a share of the examples, the top pair of
    sensors removed (zeroed and flagged in the mask). Returns readings, mask."""
    n = readings.shape[0]
    s = sigma[0] + (sigma[1] - sigma[0]) * torch.rand(n, 1, 1, generator=generator, device=DEVICE)
    r = readings + s * torch.randn(readings.shape, generator=generator, device=DEVICE)
    mask = torch.ones(n, 6, device=DEVICE)
    mask[torch.rand(n, generator=generator, device=DEVICE) < four_share, 4:] = 0
    return r * mask[:, None, :], mask

def loss(net, readings, mask, targets):
    """Mean squared error of the nine outputs, each scaled by its range."""
    return (((net(readings, mask) - targets) / SCALE) ** 2).mean()

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide:

* **what the network sees.** The raw readings are 11 to 35 K, but what the disc does is a change of a few tenths of a kelvin on top: the readings relative to the healthy slot's (`HEALTHY`) are the signal. A missing sensor must be told apart from a reading of zero (the mask);
* **what it returns, and how.** Scaled outputs (`SCALE`, `CENTRE`), and whether the bounds of the range are built in;
* **its size.** Layers, neurons, activation.

Remember part A, section 4: the readings fix the disc's strength far better than its radius and conductivity apart.

In [ ]:
# TODO 1 - your network. net(readings (n, 4, 6) [K], mask (n, 6)) -> (n, 9), the contract's order and units
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, readings, mask):
        ...                                              # return (n, 9)

torch.manual_seed(72)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Simulate your training set with `make_data` (decide how many slots: about 30 ms each on a CPU, and the simulation is part of the offline cost), then train `net` with `noisy` and `loss`. Decide the optimiser, the epochs, the batch size, the learning rate and its schedule, and whether to keep slots apart for validation. Keep the loss history for TODO 3, and **measure the offline time**, simulation included: set `TRAIN_SECONDS`.

`quick_score()` below gives, over the five test cases, the worst centre error [mm], the worst strength error [%] and the worst hot spot error at rated current [K].

In [ ]:
# A quick score, given: the worst errors over the five test cases
G = np.load("MP7.1B_truth.npz")
CASES = [str(c) for c in G["cases"]]
CASE_R = torch.tensor(np.stack([G[f"case{n}_readings"] for n in CASES]), device=DEVICE)   # (5, 4, 6) [K]
CASE_M = torch.tensor(np.stack([G[f"case{n}_mask"] for n in CASES]), device=DEVICE)       # (5, 6)

def answer(readings=CASE_R, mask=CASE_M):
    """The network's answer to sets of readings: (n, 9) as a NumPy array."""
    with torch.no_grad():
        return net(readings, mask).cpu().numpy()

def case_errors(p):
    """Per case: the centre error [mm], the strength error [-] and the hot spot error at rated current [K]."""
    out = []
    for i, n in enumerate(CASES):
        d = G[f"case{n}_disc"]
        out.append((np.hypot(p[i, 0] - d[0], p[i, 1] - d[1]) * 1e3,
                    p[i, 4] / mp.strength(d[2], d[3]) - 1, p[i, 7] - G[f"case{n}_hot"][2]))
    return np.array(out)

def quick_score():
    e = case_errors(answer())
    return np.abs(e[:, 0]).max(), np.abs(e[:, 1]).max() * 100, np.abs(e[:, 2]).max()

In [ ]:
# TODO 2 - your training
history = []
t0 = time.perf_counter()
...                                                      # simulate your training set, then train; append the loss to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"offline {TRAIN_SECONDS:.0f} s; worst errors: centre %.2f mm, strength %.0f %%, hot spot %.2f K" % quick_score())

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss during training, and the network's answers against the truth on held-out slots (centre, strength, hot spot). Save each figure you want in the report as `MP7.1B_report_<name>.png` (in `cc.output_path`).

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores and keeps the answers in `R`: its answers to the five test cases, its answers to 100 fresh noisy sets of readings of case 1 (to see the spread the noise causes), and the **online time** for 1, 10 and 100 new sets of readings, against a new least-squares fit (part C's time, measured on this runtime).

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
out = {"cases": np.array(CASES), "answer": answer()}                   # (5, 9)
# 100 new sets of readings of case 1: the same disc, fresh noise of its σ
rng = np.random.default_rng(100)
clean = G["case1_clean"]
rep = clean[None] + float(G["case1_noise"]) * rng.standard_normal((100,) + clean.shape)
out["case1_repeat"] = answer(torch.tensor(rep, device=DEVICE), torch.ones(100, 6, device=DEVICE))
timing = {}
for k in (1, 10, 100):
    r = torch.tensor(rep[:k], device=DEVICE); m = torch.ones(k, 6, device=DEVICE)
    timing[f"net_{k}"] = middle(lambda: answer(r, m))
out.update({k: np.array(v) for k, v in timing.items()})
out["lsq_seconds"] = np.array(REF["t_ref_s"])
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP7.1B_net.pt"))
print(f"online, one new set of readings: network {timing['net_1']*1e3:.2f} ms, least squares {REF['t_ref_s']*1e3:.0f} ms "
      f"({REF['t_ref_s']/timing['net_1']:.0f} times)")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the hidden values in the truth file `G` and the least-squares reference `REF` of part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the ten questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP7.1B_truth.npz")
print("in R:", ", ".join(R))
print(f"least squares (part C): {REF['t_ref_s']:.1f} s per new set of readings")

## E2 · TODO 4 · The goal metrics

Compute, for each of the five cases, and put in `METRICS` (a name and a number each):

* the centre error [mm], the strength error [%] and the hot spot error at rated current [K], the network's and the least-squares reference's (`REF["lsq"]`);
* the network's online time for one set of readings, the least-squares time, and their ratio;
* the offline time.

Then say in one line each whether the four requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>50}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP7.1B_report_<name>.png`, the figures your answers refer to. At least: each case's true disc, the network's and the least-squares' on the slot; the spread of the network's answers to the 100 noisy repeats of case 1 (`R["case1_repeat"]`); and the online times for 1, 10 and 100 sets of readings against least squares.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the ten questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 5 are about your network and its result; 6 to 9 ask you to use it; 10 concludes. The next cell writes `MP7.1B_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its inputs (and any features computed from the readings), layers, neurons, outputs and number of weights. Why this design?": """
""",
    "2. Your training data and training: how many slots you simulated and how long that took, the noise and missing sensors, the optimiser, epochs and the offline time in seconds.": """
""",
    "3. The five test cases: give the network's centre error [mm], strength error [%] and hot spot error at rated current [K] for each. Which requirements are met?": """
""",
    "4. The same three errors for the least-squares reference: on which cases is the network better, on which worse, and why?": """
""",
    "5. The online time for one new set of readings: the network against a new least-squares fit, in milliseconds, and the ratio. Then for 10 and 100 sets: how do the two grow?": """
""",
    "6. Case 1 repeated 100 times with fresh noise: how large is the spread of the network's centre and hot spot? Compare it with the Cramér-Rao bound of part C.": """
""",
    "7. The radius and the conductivity apart: give the network's and least squares' r_d and k_d for each case against the truth. Why is the strength better determined than either?": """
""",
    "8. Which case is hardest, and why: its disc's position relative to the sensors, its strength, the noise, the missing pair? Use the held-out slots of your training to say which discs your network cannot find.": """
""",
    "9. Choose one case and fit it with an inverse physics-informed network (θ(x, y, I) and k(x, y), fitted to the readings and the equation), or argue with numbers why it would be slower: what is its online cost for one new set of readings?": """
""",
    "10. To conclude: an operator inspects 500 machines a year. Which method would you give them, given your measured accuracy, online time and offline time? What would change if the sensors were placed differently?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" for k, v in METRICS.items()]
    out = ["# MP7.1B — How Old Is the Insulation?", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP7.1B_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP7.1B_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP7.1B_report.md into MP7.1B_report.pdf, with any figure
# saved as MP7.1B_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP7.1B_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP7.1B_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP7.1B_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP7.1B_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass